# 01 - Data Quality - Germany Used Cars 2023

**Project question:** do electric cars lose value faster than combustion cars, *at the same age and mileage*?
**Extra metric:** value retention by brand and fuel type.

Before any analysis we answer one question: **can we trust this data?**
This notebook looks at the raw AutoScout24 file, writes down every problem, and justifies each cleaning decision.
The final code lives in `src/cleaning.py` so notebooks 02, 03 and the Streamlit app reuse the exact same cleaning.

| Part | Content |
|---|---|
| 1 | First look at the raw file |
| 2 | Broken rows (shifted columns) |
| 3 | Missing values - *why* are they missing? |
| 4 | Duplicates |
| 5 | Impossible or suspicious values |
| 6 | Fuel types, dates and age |
| 7 | Clean with `src/cleaning.py` + validate |

## Setup

In [1]:
import sys
sys.path.append("..")            # so we can import from src/

import numpy as np
import pandas as pd
import plotly.express as px

from src.cleaning import (load_raw, find_shifted_rows, parse_types, clean,
                          VALID_FUELS, MIN_PRICE, MAX_MILEAGE, REFERENCE_DATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)

In [2]:
raw = load_raw()
print("Raw file:", raw.shape)

Raw file: (251079, 14)


---
# 1. First look at the dataset

In [3]:
df = raw.copy()          # always keep the raw data untouched!
df.head()

,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description
0,alfa-romeo,Alfa Romeo GTV,red,10/1995,1995,1300,148,201,Manual,Petrol,"10,9 l/100 km",260 g/km,160500.0,2.0 V6 TB
1,alfa-romeo,Alfa Romeo 164,black,02/1995,1995,24900,191,260,Manual,Petrol,NaN,- (g/km),190000.0,"Q4 Allrad, 3.2L GTA"
2,alfa-romeo,Alfa Romeo Spider,black,02/1995,1995,5900,110,150,Unknown,Petrol,NaN,- (g/km),129000.0,ALFA ROME 916
3,alfa-romeo,Alfa Romeo Spider,black,07/1995,1995,4900,110,150,Manual,Petrol,"9,5 l/100 km",225 g/km,189500.0,2.0 16V Twin Spark L
4,alfa-romeo,Alfa Romeo 164,red,11/1996,1996,17950,132,179,Manual,Petrol,"7,2 l/100 km",- (g/km),96127.0,"3.0i Super V6, absoluter Topzustand !"


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 251079 entries, 0 to 251078
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   brand                     251079 non-null  str    
 1   model                     251079 non-null  str    
 2   color                     250913 non-null  str    
 3   registration_date         251075 non-null  str    
 4   year                      251079 non-null  str    
 5   price_in_euro             251079 non-null  str    
 6   power_kw                  250945 non-null  str    
 7   power_ps                  250950 non-null  str    
 8   transmission_type         251079 non-null  str    
 9   fuel_type                 251079 non-null  str    
 10  fuel_consumption_l_100km  224206 non-null  str    
 11  fuel_consumption_g_km     251079 non-null  str    
 12  mileage_in_km             250927 non-null  float64
 13  offer_description         251078 non-null  str    
dtyp

In [5]:
df.isna().mean().mul(100).round(1).sort_values(ascending=False)

fuel_consumption_l_100km    10.7
color                        0.1
power_kw                     0.1
power_ps                     0.1
mileage_in_km                0.1
brand                        0.0
model                        0.0
registration_date            0.0
year                         0.0
price_in_euro                0.0
transmission_type            0.0
fuel_type                    0.0
fuel_consumption_g_km        0.0
offer_description            0.0
dtype: float64

**Surprises** (our cleaning to-do list):
- The task said ~200k listings, the file has **251,079** rows.
- `year`, `price_in_euro`, `power_kw`, `power_ps` are **text** (`str`), not numbers -> something non-numeric is hiding in them.
- Only `mileage_in_km` is numeric, and it has a max of 3.8 **million** km.
- `fuel_consumption_l_100km` is **10.7% empty** and mixes units (`l/100 km`, `kWh/100 km`, `km (Ort)`...).
- The first column is an unnamed old index.

---
# 2. Broken rows: the scraper shifted the columns

Why would a *numeric* column be text? Let's look at the values that are not numbers.

In [6]:
for col in ["year", "price_in_euro", "power_kw", "power_ps"]:
    not_numeric = pd.to_numeric(df[col], errors="coerce").isna() & df[col].notna()
    print(f"{col:14s} non-numeric values: {not_numeric.sum():4d} | examples: {df.loc[not_numeric, col].head(4).tolist()}")

year           non-numeric values:  195 | examples: ['Petrol', 'Automatic', 'Manual', 'Petrol']
price_in_euro  non-numeric values:  199 | examples: [' KLIMAAUTOMATIK*', 'Automatik', '6-Gang', ' XEN AAC SHZ FSP APS 5TRG']


power_kw       non-numeric values:  159 | examples: ['7,8 l/100 km', '5,4 l/100 km', '4,6 l/100 km', '121 g/km']


power_ps       non-numeric values:   81 | examples: ['194 g/km', '126 g/km', '4,6 l/100 km', '4,9 l/100 km']


In [7]:
df["fuel_type"].value_counts().head(16)

fuel_type
Petrol           143280
Diesel            86421
Hybrid            12607
Electric           5967
LPG                1255
CNG                 508
Diesel Hybrid       476
Other               178
Unknown              96
Hydrogen             82
Automatic            26
Manual               18
01/2023              13
Ethanol              10
02/2023               5
03/2017               3
Name: count, dtype: int64

`fuel_type` should be a fuel, but we find `Automatic`, `Manual`, `01/2023`... In these rows the **values moved to the wrong columns**.
Let's see a few of them.

In [8]:
shifted = find_shifted_rows(df)
print("Shifted rows:", shifted.sum(), f"({shifted.mean():.2%} of the file)")
df[shifted].head(6)

Shifted rows: 199 (0.08% of the file)


,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description
2400,audi,Audi A4,grey,Manual,Petrol,KLIMAAUTOMATIK*,"7,8 l/100 km",75,Manual,07/2004,187 g/km,2004,2460.0,Avant 1.6 *TÜV 09/23
2743,audi,Audi A4,silver,8449,Automatic,Automatik,2005,194 g/km,Automatic,74.194 km,Petrol,11/2005,NaN,Lim. 2.0
5297,audi,Audi A1,blue,8999,Manual,6-Gang,2011,126 g/km,Manual,110.250 km,Petrol,04/2011,NaN,1.4 TFSI Ambition
9616,audi,Audi A1,white,Manual,Petrol,XEN AAC SHZ FSP APS 5TRG,"5,4 l/100 km",90,Manual,06/2014,128 g/km,2014,12250.0,Sportback Sportline
12206,audi,Audi A6,black,Manual,Diesel,NAVplus LEDER PANO ALC,"4,6 l/100 km",140,Manual,12/2016,119 g/km,2016,21250.0,Avant 2.0 TDI ultra
13554,audi,Audi A5,grey,99.900 km,04/2017,amp,121 g/km,"4,6 l/100 km",Automatic,Automatic,2017,Diesel,NaN,Sportback qu S-LINE PANO B&amp


**Decision: drop them.** The shift is *not* consistent (the columns move differently from row to row, sometimes the price is lost inside the
description), so we cannot repair them safely. We lose only **0.08%** of the data.
Is that loss random? Let's check it does not hit one brand or fuel hard.

In [9]:
by_brand = pd.DataFrame({"shifted": df[shifted]["brand"].value_counts(),
                         "all": df["brand"].value_counts()}).fillna(0)
by_brand["share_%"] = (by_brand["shifted"] / by_brand["all"] * 100).round(2)
by_brand.sort_values("share_%", ascending=False).head(6)

,shifted,all,share_%
brand,,,
daihatsu,1.0,204,0.49
porsche,17.0,3958,0.43
toyota,33.0,8228,0.40
skoda,20.0,14039,0.14
nissan,4.0,3784,0.11
opel,18.0,20388,0.09


Spread over many brands, none above 0.5% of its own rows. Safe to drop.
Now that the broken rows are out, we can convert the types.

In [10]:
df = parse_types(df[~shifted])
df[["year", "price_in_euro", "power_kw", "power_ps", "registration_date"]].dtypes

year                          int64
price_in_euro                 int64
power_kw                    float64
power_ps                    float64
registration_date    datetime64[us]
dtype: object

In [11]:
# year and registration_date are two copies of the same information: are they consistent?
print("Rows where year != registration year:", (df["year"] != df["registration_date"].dt.year).sum())
print("Latest registration:", df["registration_date"].max().date())

Rows where year != registration year: 0
Latest registration: 2023-11-01


`year` always matches `registration_date`, so we keep `registration_date` (it has the **month**, a finer age).
The newest registration is **11/2023**, so we assume the data was scraped in November 2023 and compute `age_years` from **2023-11-01**.

---
# 3. Missing values - *why* are they missing?

In [12]:
missing = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing = missing[missing > 0].reset_index()
missing.columns = ["column", "missing_%"]

fig = px.bar(missing, x="missing_%", y="column", orientation="h", text="missing_%",
             title="Missing values per column (%)")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

Only `fuel_consumption_l_100km` has a real hole (~11%). Is it **random**, or does it depend on another column (MAR)?

In [13]:
miss_by_fuel = (df.groupby("fuel_type")["fuel_consumption_l_100km"]
                  .agg(rows="size", missing_pct=lambda s: s.isna().mean() * 100).round(1)
                  .sort_values("rows", ascending=False))
miss_by_fuel

,rows,missing_pct
fuel_type,,
Petrol,143280,7.7
Diesel,86421,8.6
Hybrid,12607,17.9
Electric,5967,92.1
LPG,1255,14.7
CNG,508,18.5
Diesel Hybrid,476,20.8
Other,178,51.1
Unknown,96,76.0


**Insight:** it is **MAR, not random**. Electric cars are **~92% missing** because their consumption is in kWh (or in range, `km (Ort)`), not l/100 km.
So we **must not** compare consumption across fuels - and we do not need it: our hypothesis uses *price, age, mileage and fuel*.
**Decision: do not use `fuel_consumption_*`.** The other columns have < 0.2% missing:

In [14]:
df[["color", "power_kw", "power_ps", "mileage_in_km"]].isna().sum()

color            166
power_kw         128
power_ps         128
mileage_in_km     62
dtype: int64

- `color`, `power_*`: not needed for the hypothesis -> leave them as they are.
- `mileage_in_km`: **needed** (it is one of our controls) -> we drop the few rows without it (~60).

---
# 4. Duplicates

First decide what makes a row unique. There is **no listing id**, so we use *all columns* as the key.

In [15]:
dups = df.duplicated(keep=False)
print("Rows involved in an exact duplicate:", dups.sum())
print("Rows we would remove (keep first):  ", df.duplicated().sum(), f"({df.duplicated().mean():.1%})")
df[dups].sort_values(["brand", "model", "registration_date", "mileage_in_km"]).head(6)

Rows involved in an exact duplicate: 10380
Rows we would remove (keep first):   6338 (2.5%)


,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description
106,alfa-romeo,Alfa Romeo 147,silver,2003-10-01,2003,1550,88.0,120.0,Manual,Petrol,"8,2 l/100 km",- (g/km),197000.0,1.6 Twin Spark Impression
107,alfa-romeo,Alfa Romeo 147,silver,2003-10-01,2003,1550,88.0,120.0,Manual,Petrol,"8,2 l/100 km",- (g/km),197000.0,1.6 Twin Spark Impression
173,alfa-romeo,Alfa Romeo 159,grey,2006-09-01,2006,3800,147.0,200.0,Automatic,Diesel,"8,1 l/100 km",215 g/km,210000.0,Sportwagon 2.4 JTDM 20V DPF Automati...
174,alfa-romeo,Alfa Romeo 159,grey,2006-09-01,2006,3800,147.0,200.0,Automatic,Diesel,"8,1 l/100 km",215 g/km,210000.0,Sportwagon 2.4 JTDM 20V DPF Automati...
562,alfa-romeo,Alfa Romeo Giulia,black,2016-10-01,2016,21890,132.0,179.0,Automatic,Diesel,"4,2 l/100 km",109 g/km,59900.0,Super + RFK+DAB+SHZ+XENON+LEDER+ NAV...
566,alfa-romeo,Alfa Romeo Giulia,black,2016-10-01,2016,21890,132.0,179.0,Automatic,Diesel,"4,2 l/100 km",109 g/km,59900.0,Super + RFK+DAB+SHZ+XENON+LEDER+ NAV...


**2.5% of the file are exact copies** (same brand, model, date, price, mileage, description...). Two different cars matching *to the exact km and
the exact description* is not plausible -> it is the **same listing scraped twice**.
**Decision: drop exact duplicates** (otherwise some cars weigh double in every average).

---
# 5. Impossible or suspicious values

In [16]:
df[["price_in_euro", "mileage_in_km", "power_kw", "power_ps"]].describe().round(0)

,price_in_euro,mileage_in_km,power_kw,power_ps
count,250880.0,250818.0,250752.0,250752.0
mean,26138.0,85369.0,126.0,172.0
std,36988.0,78721.0,72.0,98.0
min,120.0,0.0,1.0,1.0
25%,11990.0,24950.0,82.0,111.0
50%,19890.0,67515.0,110.0,150.0
75%,29950.0,126538.0,140.0,190.0
max,5890500.0,3800000.0,735.0,999.0


### 5.1 Price

In [17]:
counts, edges = np.histogram(df["price_in_euro"], bins=100)      # bin first: lighter than sending 250k rows to the browser
price_hist = pd.DataFrame({"price_in_euro": edges[:-1], "listings": counts})
fig = px.bar(price_hist, x="price_in_euro", y="listings", log_y=True, title="Price distribution (log scale on counts)")
fig.show()
print("price < 500 EUR:   ", (df["price_in_euro"] < MIN_PRICE).sum())
print("price > 300,000 EUR:", (df["price_in_euro"] > 300_000).sum())

price < 500 EUR:    103
price > 300,000 EUR: 591


In [18]:
print("Cheapest listings:")
display(df.nsmallest(5, "price_in_euro")[["brand", "model", "registration_date", "mileage_in_km", "price_in_euro", "offer_description"]])
print("Most expensive listings:")
display(df.nlargest(5, "price_in_euro")[["brand", "model", "registration_date", "mileage_in_km", "price_in_euro"]])

Cheapest listings:


,brand,model,registration_date,mileage_in_km,price_in_euro,offer_description
213798,volkswagen,Volkswagen Polo,1998-11-01,200000.0,120,Polo 50
168576,renault,Renault Espace,2004-01-01,237500.0,149,"Expression IV Motorschaden, Teile au..."
58531,ford,Ford Ka/Ka+,2006-02-01,103000.0,150,Ka 1.3
131,alfa-romeo,Alfa Romeo 147,2004-01-01,189000.0,200,147 1.6 Twin Spark Progression
103333,mercedes-benz,Mercedes-Benz A 140,2002-08-01,170000.0,200,A-Klasse Classic


Most expensive listings:


,brand,model,registration_date,mileage_in_km,price_in_euro
93699,lamborghini,Lamborghini,2018-08-01,65.0,5890500
93840,lamborghini,Lamborghini,2021-11-01,177.0,3250000
106583,mercedes-benz,Mercedes-Benz SLR,2007-01-01,15793.0,2490000
1509,aston-martin,Aston Martin,2011-08-01,8900.0,2289000
165445,porsche,Porsche 918,2014-06-01,5125.0,1990000


- Prices under **500 €** (103 rows): not realistic sale prices for a car on a dealer platform (placeholders, damaged cars, deposits). **Drop.**
- Prices above 300k € are **real**: Aston Martin, Ferrari, Bentley... Domain knowledge decides, not pandas -> **keep**. They will be handled with the
  **median** and with *ratios* (retention), not raw means.

### 5.2 Mileage

In [19]:
print("mileage == 0:      ", (df["mileage_in_km"] == 0).sum())
print("mileage > 500,000: ", (df["mileage_in_km"] > MAX_MILEAGE).sum())
display(df.nlargest(5, "mileage_in_km")[["brand", "model", "registration_date", "mileage_in_km", "price_in_euro"]])

mileage == 0:       202
mileage > 500,000:  131


,brand,model,registration_date,mileage_in_km,price_in_euro
62477,ford,Ford Transit,2014-10-01,3800000.0,9600
57764,ford,Ford Galaxy,2003-11-01,2830000.0,1600
57953,ford,Ford Fiesta,2004-02-01,2580000.0,1500
138830,opel,Opel Corsa,2003-01-01,2390000.0,500
56269,fiat,Fiat Doblo,2022-06-01,2300000.0,650


In [20]:
# Is mileage == 0 a data error? Look at the registration year of those cars
df.loc[df["mileage_in_km"] == 0, "registration_date"].dt.year.value_counts().head(5)

registration_date
2023    166
2022     30
2021      3
2016      1
2020      1
Name: count, dtype: int64

- `mileage == 0` are almost all **2022-2023 cars** (new / pre-registered) -> plausible, **keep**.
- Above **500,000 km** (131 rows): a typing mistake (3.8 million km!) or a taxi on its 3rd engine -> either way it would distort the age-mileage control. **Drop.**

### 5.3 Power: kW vs PS consistency

In [21]:
ratio = (df["power_ps"] / df["power_kw"]).dropna()
print(ratio.describe().round(3))
print("Cars above 600 kW:", (df["power_kw"] > 600).sum())
df.loc[df["power_kw"] > 600, ["brand", "model", "power_kw", "fuel_type"]].drop_duplicates("model").head(5)

count    250752.000
mean          1.360
std           0.003
min           1.000
25%           1.358
50%           1.360
75%           1.364
max           1.400
dtype: float64
Cars above 600 kW: 53


,brand,model,power_kw,fuel_type
15858,audi,Audi RS3,625.0,Petrol
18228,audi,Audi RS7,625.0,Petrol
51518,dodge,Dodge Challenger,626.0,Petrol
52334,ferrari,Ferrari 812,618.0,Petrol
52421,ferrari,Ferrari SF90 Spider,735.0,Hybrid


`PS / kW` is always ~ **1.36** (the real conversion factor): the two columns agree. The >600 kW cars are Ferrari, Audi RS, Dodge... real. **Nothing to clean.**

---
# 6. Fuel types, dates and age

In [22]:
fuel = df["fuel_type"].value_counts().rename("rows").to_frame()
fuel["share_%"] = (fuel["rows"] / fuel["rows"].sum() * 100).round(2)
fuel

,rows,share_%
fuel_type,,
Petrol,143280,57.11
Diesel,86421,34.45
Hybrid,12607,5.03
Electric,5967,2.38
LPG,1255,0.50
CNG,508,0.20
Diesel Hybrid,476,0.19
Other,178,0.07
Unknown,96,0.04


**Group sizes matter for the hypothesis:**
- **Electric: 5,967 listings (2.4%)** - enough to compare, but they are *young* (see below).
- `Hybrid` and `Diesel Hybrid` are two labels for the same idea -> merge as **Hybrid**.
- `LPG`, `CNG`, `Ethanol`, `Hydrogen`, `Other`, `Unknown`: tiny groups -> **Other** (not compared one by one).

Our comparison is therefore **Electric vs Petrol vs Diesel** (+ Hybrid as extra).

In [23]:
# The most important check for the hypothesis: are EVs and combustion cars comparable in AGE?
tmp = df.assign(age_years=(REFERENCE_DATE - df["registration_date"]).dt.days / 365.25,
                is_ev=df["fuel_type"].eq("Electric"))
tmp = tmp[tmp["fuel_type"].isin(["Electric", "Petrol", "Diesel"])]

bins = np.arange(0, 31, 1)
age_hist = (tmp.groupby("fuel_type")["age_years"]
               .apply(lambda s: pd.Series(np.histogram(s, bins=bins)[0] / len(s) * 100, index=bins[:-1]))
               .rename("share_%").reset_index().rename(columns={"level_1": "age_years"}))
fig = px.bar(age_hist, x="age_years", y="share_%", color="fuel_type", barmode="overlay", opacity=0.6,
             title="Age distribution by fuel type (% of each group)", labels={"age_years": "age (years)"})
fig.show()
tmp.groupby("fuel_type")["age_years"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
fuel_type,,,,,,,,
Diesel,86421.0,7.6,4.6,0.3,4.3,6.7,10.3,28.8
Electric,5967.0,2.6,1.9,0.0,1.0,2.2,3.5,22.8
Petrol,143280.0,7.4,5.9,0.3,3.1,5.7,10.7,28.8


**Insight (key for notebook 03):** electric cars are **young** - median 2.2 years, **95% are 6 years old or less**. Petrol and diesel go back to 1995 (median 5.7 and 6.7 years).
A naive average price comparison would mix *fuel* with *age*. That is exactly why the hypothesis says "at the same age and mileage":
we will compare only where the age ranges **overlap** (roughly 0-6 years).

---
# 7. Clean with `src/cleaning.py` and validate

All the decisions above live in one function, so every notebook and the app get the same data.

In [24]:
clean_df, log = clean(raw, return_log=True)
log

,step,rows_left,rows_removed
0,raw file,251079,0
1,remove shifted rows,250880,199
2,remove exact duplicates,244542,6338
3,remove price < 500 EUR,244439,103
4,"remove mileage > 500,000 km",244308,131
5,remove rows without mileage,244255,53


In [25]:
fig = px.bar(log.iloc[1:], x="rows_removed", y="step", orientation="h", text="rows_removed",
             title="Rows removed at each cleaning step")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()
print(f"Raw {len(raw):,} -> clean {len(clean_df):,} rows  ({1 - len(clean_df)/len(raw):.1%} removed)")

Raw 251,079 -> clean 244,255 rows  (2.7% removed)


`assert` stops the code when a condition is false - a free safety net:

In [26]:
assert clean_df["price_in_euro"].between(MIN_PRICE, 6_000_000).all()
assert clean_df["mileage_in_km"].between(0, MAX_MILEAGE).all()
assert clean_df["fuel_type"].isin(VALID_FUELS).all()
assert not clean_df.duplicated().any()
assert clean_df["age_years"].ge(0).all()
assert clean_df[["price_in_euro", "mileage_in_km", "age_years", "fuel_group"]].notna().all().all()
print("All checks passed ")

All checks passed 


In [27]:
clean_df.groupby("fuel_group").agg(listings=("price_in_euro", "size"),
                                   median_price=("price_in_euro", "median"),
                                   median_age=("age_years", "median"),
                                   median_km=("mileage_in_km", "median")).round(1)

,listings,median_price,median_age,median_km
fuel_group,,,,
Diesel,84738,19900.0,6.8,109497.5
Electric,5870,29990.0,2.2,12391.0
Hybrid,12214,30990.0,2.2,16768.5
Other,2012,16799.5,5.0,58000.0
Petrol,139421,17990.0,5.8,57000.0


## Summary

| Problem | Where | Decision |
|---|---|---|
| Columns shifted by the scraper | 199 rows | drop |
| Text instead of numbers | `year`, `price_in_euro`, `power_*` | convert after dropping broken rows |
| Exact duplicate listings | 6,338 rows (2.5%) | drop |
| Price < 500 € | 103 rows | drop |
| Mileage > 500,000 km | 131 rows | drop |
| Missing mileage | 53 rows (after the previous steps) | drop (it is a control variable) |
| `fuel_consumption_*` missing for EVs (MAR) and mixed units | ~11% | **do not use** |
| Prices > 300k €, power > 600 kW, mileage = 0 | real luxury / new cars | keep |
| EVs are young (95% <= 6 years) | key for the hypothesis | compare only on the **age overlap** |

Next: `02_eda.ipynb` - explore the clean data (`clean_df`) from `src.cleaning.clean`.